# LLM-Assisted Environmental-Economic Indicator Coding

This notebook classifies indicators using the attached codebook and example indicators.

Input workbook:

`LLM_indicator_coding_workflow.xlsx`

Input sheets:

1. `Step1_Input_Indicators`
2. `Step2_Codebook`
3. `Step3_Examples_Gold_Set`

Output sheet:

4. `Step4_LLM_Output`

The workflow stops after creating or replacing `Step4_LLM_Output`. No human validation, reliability analysis, or final adjudication is performed in this notebook.

## Stage 1. Set project folder

This stage sets the working directory and checks that the workbook is available.

Change `PROJECT_DIR` to the folder where you saved:

`LLM_indicator_coding_workflow.xlsx`

In [1]:
import os
from pathlib import Path

# Change this path to your actual folder
PROJECT_DIR = r"D:\School\Fall 2024\DDL\ADB\00_Revision\Data_Categories"

os.chdir(PROJECT_DIR)

WORKBOOK_PATH = "LLM_indicator_coding_workflow.xlsx"

print("Current working directory:")
print(os.getcwd())

print("\nFiles in this folder:")
for file in os.listdir():
    print(" -", file)

if not Path(WORKBOOK_PATH).exists():
    raise FileNotFoundError(f"Cannot find {WORKBOOK_PATH}. Please place it in the project folder.")
else:
    print(f"\nWorkbook found: {WORKBOOK_PATH}")

Current working directory:
D:\School\Fall 2024\DDL\ADB\00_Revision\Data_Categories

Files in this folder:
 - .ipynb_checkpoints
 - 931AC240
 - anaconda_projects
 - categorization.ipynb
 - LLM_indicator_coding_workflow.xlsx

Workbook found: LLM_indicator_coding_workflow.xlsx


## Stage 2. Install and import packages

This stage installs and imports the packages needed to read Excel files, call the LLM, and write the output sheet.

In [2]:
%pip install pandas openpyxl openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import time
import pandas as pd
from openpyxl import load_workbook
from openai import OpenAI

## Stage 3. Add API key and model name

Paste your OpenAI API key in the code cell below.

For the model name, use the model that is available in your API account. If `gpt-5.5` is not available in your account, replace it with another available model.

In [4]:

import os

# Paste your API key here
os.environ["OPENAI_API_KEY"] = "TYPEHERE"

# Change this if needed based on your API access
MODEL_NAME = "gpt-5.5"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != "PASTE_YOUR_API_KEY_HERE":
    print("API key loaded.")
else:
    print("Please paste your OpenAI API key before running the LLM coding stage.")

API key loaded.


## Stage 4. Load workbook sheets

This stage loads the input indicator sheet, codebook sheet, and example coding sheet.

The code is revised to match the actual workbook structure.

In [5]:
INPUT_SHEET = "Step1_Input_Indicators"
CODEBOOK_SHEET = "Step2_Codebook"
EXAMPLES_SHEET = "Step3_Examples_Gold_Set"
OUTPUT_SHEET = "Step4_LLM_Output"

indicators_df = pd.read_excel(WORKBOOK_PATH, sheet_name=INPUT_SHEET)
codebook_df = pd.read_excel(WORKBOOK_PATH, sheet_name=CODEBOOK_SHEET, header=None)
examples_df = pd.read_excel(WORKBOOK_PATH, sheet_name=EXAMPLES_SHEET)

# Clean column names for indicators and examples
indicators_df.columns = [str(col).strip() for col in indicators_df.columns]
examples_df.columns = [str(col).strip() for col in examples_df.columns]

print("Indicators sheet preview:")
display(indicators_df.head())

print("\nCodebook sheet preview:")
display(codebook_df.head(20))

print("\nExamples sheet preview:")
display(examples_df.head())

Indicators sheet preview:


,no,indicator,Definition,unit,source,original,dataset
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb



Codebook sheet preview:


,0,1,2,3
0,Stage 1: Assign exactly one primary category,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,Primary category,Definition,Main coding question,Typical keywords
3,Stock (S),A measurable quantity that persists over time ...,Does the indicator measure a quantity that exi...,"area, capital, assets, reserves, resources, st..."
4,Flow (F),"A movement, transformation, service, income, e...",Does the indicator measure something happening...,"emissions, expenditure, revenue, generation, t..."
5,Action (A),"A concrete policy instrument, institutional ru...",Does the indicator describe an actual policy i...,"carbon tax, environmental tax, expenditure pro..."
6,Policy Guide (P),"A pledge, target, plan, aspiration, scenario, ...","Does the indicator describe a target, commitme...","target, pledge, Nationally Determined Contribu..."
7,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN
9,Stage 2: Assign boundary flags only for Stock ...,NaN,NaN,NaN



Examples sheet preview:


,No,Indicator,Description,Reviewed primary category,Reviewed boundary flags,Explanation,Justification,Review status,Notes
0,1,Quarterly Greenhouse Gas (GHG) Air Emissions A...,Quarterly greenhouse gas emissions by activity...,F,DDP,Emissions are unpriced releases that reduce en...,Quarterly emissions are measured over time.,Reviewed,NaN
1,2,Annual Greenhouse Gas (GHG) Air Emissions Acco...,Annual greenhouse gas emissions by activity an...,F,DDP,Emissions are degradation flows.,Annual emissions are measured over a period.,Reviewed,NaN
2,3,National Greenhouse Gas Emissions Inventories,National greenhouse gas emissions inventories ...,F,DDP,Inventories measure emissions flows.,Emissions by sector are annual flow measures.,Reviewed,NaN
3,4,"Implied National Mitigation Target, based on N...",National mitigation targets as determined by c...,P,0,These are policy targets rather than observed ...,They guide planning but do not directly measur...,Reviewed,NaN
4,5,Environmental Taxes,"A tax whose base is a physical unit, or proxy,...",A,0,Tax instruments are implemented fiscal tools.,"This is a policy instrument, not a stock or fl...",Needs review,Previous example assigned boundary. Codebook s...


## Stage 5. Check required input columns

Your workbook uses the following columns in `Step1_Input_Indicators`:

- `indicator_id`
- `indicator_name`
- `description`
- `unit`
- `source_database`
- `source_url_or_file`
- `notes`

The first two are required. The others are recommended and will be created as blank columns if missing.

In [6]:
# ============================================================
# Stage 5. Check and standardize input columns
# ============================================================

# Your actual workbook has these columns:
# no, indicator, Definition, unit, source, original, dataset

# Clean column names
indicators_df.columns = [str(col).strip() for col in indicators_df.columns]

print("Columns found in Step1_Input_Indicators:")
print(list(indicators_df.columns))

required_columns = ["indicator"]

recommended_columns = [
    "no",
    "Definition",
    "unit",
    "source",
    "original",
    "dataset"
]

missing_required = [col for col in required_columns if col not in indicators_df.columns]

if missing_required:
    raise ValueError(
        f"Missing required columns in {INPUT_SHEET}: {missing_required}. "
        f"Available columns are: {list(indicators_df.columns)}"
    )

for col in recommended_columns:
    if col not in indicators_df.columns:
        indicators_df[col] = ""

# Keep only rows with indicator names
indicators_df = indicators_df[
    indicators_df["indicator"].notna()
    & (indicators_df["indicator"].astype(str).str.strip() != "")
].copy()

# Fill missing values with blank text for cleaner prompting
for col in required_columns + recommended_columns:
    indicators_df[col] = indicators_df[col].fillna("").astype(str).str.strip()

print(f"Number of indicators to code: {len(indicators_df)}")
display(indicators_df.head())

Columns found in Step1_Input_Indicators:
['no', 'indicator', 'Definition', 'unit', 'source', 'original', 'dataset']
Number of indicators to code: 1320


,no,indicator,Definition,unit,source,original,dataset
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb


## Stage 6. Convert the codebook into prompt text

The codebook sheet has section headers and tables rather than one simple rectangular table.

This stage preserves the full codebook content as text so the LLM can use the definitions exactly as written.

In [7]:
def codebook_to_text(df):
    """
    Convert the codebook sheet into readable text while preserving section titles,
    category definitions, coding questions, typical keywords, boundary flags, and examples.
    """
    lines = []
    
    for _, row in df.iterrows():
        values = []
        for value in row.tolist():
            if pd.notna(value) and str(value).strip() != "":
                values.append(str(value).strip())
        
        if values:
            lines.append(" | ".join(values))
    
    return "\n".join(lines)


codebook_text = codebook_to_text(codebook_df)

print("Codebook text preview:")
print(codebook_text[:2500])

Codebook text preview:
Stage 1: Assign exactly one primary category
Primary category | Definition | Main coding question | Typical keywords
Stock (S) | A measurable quantity that persists over time and can be treated as an asset, capital, liability, or accumulated state variable. | Does the indicator measure a quantity that exists at a point in time or persists across time? | area, capital, assets, reserves, resources, stock, wealth, forest area, water resources
Flow (F) | A movement, transformation, service, income, expenditure, emission, extraction, use, generation, treatment, or loss measured over a period. | Does the indicator measure something happening, moving, being produced, consumed, emitted, spent, treated, or affected over time? | emissions, expenditure, revenue, generation, treatment, supply, abstraction, collection, deaths, affected, losses, production
Action (A) | A concrete policy instrument, institutional rule, fiscal tool, regulatory action, or implemented measure. | D

## Stage 7. Convert example indicators into prompt text

This stage converts the reviewed example indicators into compact few-shot examples.

The examples include:

- indicator
- description
- reviewed primary category
- reviewed boundary flags
- explanation
- justification
- review status

Rows marked `Needs review` are retained but labeled so the model knows they are not as strong as fully reviewed examples.

In [8]:
def examples_to_text(df, max_examples=30):
    """
    Convert example coding rows into text for the prompt.
    """
    expected_cols = [
        "No",
        "Indicator",
        "Description",
        "Reviewed primary category",
        "Reviewed boundary flags",
        "Explanation",
        "Justification",
        "Review status",
        "Notes"
    ]
    
    # Keep only columns that exist
    existing_cols = [col for col in expected_cols if col in df.columns]
    temp = df[existing_cols].copy()
    
    # Remove empty rows
    if "Indicator" in temp.columns:
        temp = temp[
            temp["Indicator"].notna()
            & (temp["Indicator"].astype(str).str.strip() != "")
        ].copy()
    
    temp = temp.head(max_examples)
    
    lines = []
    for _, row in temp.iterrows():
        row_parts = []
        for col in existing_cols:
            value = row[col]
            if pd.notna(value) and str(value).strip() != "":
                row_parts.append(f"{col}: {str(value).strip()}")
        if row_parts:
            lines.append(" | ".join(row_parts))
    
    return "\n".join(lines)


examples_text = examples_to_text(examples_df, max_examples=30)

print("Examples text preview:")
print(examples_text[:2500])

Examples text preview:
No: 1 | Indicator: Quarterly Greenhouse Gas (GHG) Air Emissions Accounts | Description: Quarterly greenhouse gas emissions by activity and region. | Reviewed primary category: F | Reviewed boundary flags: DDP | Explanation: Emissions are unpriced releases that reduce environmental capacity. | Justification: Quarterly emissions are measured over time. | Review status: Reviewed
No: 2 | Indicator: Annual Greenhouse Gas (GHG) Air Emissions Accounts | Description: Annual greenhouse gas emissions by activity and region. | Reviewed primary category: F | Reviewed boundary flags: DDP | Explanation: Emissions are degradation flows. | Justification: Annual emissions are measured over a period. | Review status: Reviewed
No: 3 | Indicator: National Greenhouse Gas Emissions Inventories | Description: National greenhouse gas emissions inventories by sector of origin. | Reviewed primary category: F | Reviewed boundary flags: DDP | Explanation: Inventories measure emissions flows

## Stage 8. Define strict output schema

The LLM must return only the allowed categories and boundary flags.

Allowed primary categories:

- `S`: Stock
- `F`: Flow
- `A`: Action
- `P`: Policy Guide

Allowed boundary flags:

- `MS-NMS`
- `DDP`
- `ADE`
- `HPS`
- `0`

For Action and Policy Guide, the boundary flag should normally be `0`.

In [9]:
indicator_coding_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "primary_category": {
            "type": "string",
            "enum": ["S", "F", "A", "P"]
        },
        "primary_category_label": {
            "type": "string",
            "enum": ["Stock", "Flow", "Action", "Policy Guide"]
        },
        "boundary_flags": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["MS-NMS", "DDP", "ADE", "HPS", "0"]
            }
        },
        "confidence": {
            "type": "number",
            "minimum": 0,
            "maximum": 1
        },
        "reasoning_summary": {
            "type": "string"
        },
        "human_review_needed": {
            "type": "boolean"
        }
    },
    "required": [
        "primary_category",
        "primary_category_label",
        "boundary_flags",
        "confidence",
        "reasoning_summary",
        "human_review_needed"
    ]
}

## Stage 9. Define the expert system prompt

This prompt uses the codebook and example indicators from the workbook.

The LLM is instructed to act as a professor and subject-matter expert in environmental-economic accounting, natural capital accounting, environmental statistics, and System of Environmental-Economic Accounting indicator classification.

In [10]:
SYSTEM_PROMPT = f"""
You are a professor and subject-matter expert in environmental-economic accounting, natural capital accounting, and environmental statistics.
You have deep expertise in the System of Environmental-Economic Accounting (SEEA), environmental indicator systems, and stock-flow classification.

Use your expertise to code each indicator according to the codebook and example indicators provided below.
Apply the definitions strictly, assign exactly one primary category, and provide boundary flags only when required.
Your task is not to create new categories, but to apply the existing coding framework consistently and defensibly.

Primary categories:
- S: Stock
- F: Flow
- A: Action
- P: Policy Guide

Boundary flags:
- MS-NMS: Market Services and Near-Market Services
- DDP: Depreciation or Degradation Process
- ADE: Avoided Defensive Expenditure
- HPS: Household Produced Services
- 0: Not applicable

Important coding rules:
1. Assign exactly one primary category.
2. Use only S, F, A, or P for the primary category.
3. Use only MS-NMS, DDP, ADE, HPS, or 0 for boundary flags.
4. Assign boundary flags only for Stock and Flow indicators.
5. For Action and Policy Guide indicators, boundary_flags should normally be ["0"].
6. For Stock and Flow indicators, assign one or more boundary flags if applicable.
7. If the indicator is ambiguous, choose the best category and set human_review_needed to true.
8. Use confidence below 0.75 when the case is uncertain, ambiguous, or depends heavily on interpretation.
9. Provide a short reasoning summary that explains the coding decision.
10. Do not use category M. The only valid primary categories are S, F, A, and P.

Codebook:
{codebook_text}

Example indicators:
{examples_text}
"""

## Stage 10. Build the individual indicator prompt

This stage defines how each indicator is sent to the LLM.

The prompt uses the actual workbook columns:

- `indicator_name`
- `description`
- `unit`

In [14]:
# ============================================================
# Stage 10. Build the individual indicator prompt
# ============================================================

def build_user_prompt(row):
    """
    Build the prompt for one indicator using only the fields allowed
    for classification: indicator name, definition, and unit.
    """
    return f"""
Classify the following indicator using the codebook and examples.

Use only the following fields to determine the classification:
1. Indicator name
2. Definition
3. Unit

Do not use source, original ID, dataset name, or any other metadata
to determine the classification.

Indicator name:
{row.get("indicator", "")}

Definition:
{row.get("Definition", "")}

Unit:
{row.get("unit", "")}
"""

## Stage 11. Define the LLM classification function

This function sends one indicator to the LLM and returns structured coding output.

It includes retries in case the API call temporarily fails.

In [15]:
def classify_indicator_with_llm(row, max_retries=3, sleep_seconds=2):
    """
    Classify one indicator using the LLM.
    Returns a dictionary with structured coding results.
    """
    user_prompt = build_user_prompt(row)

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "indicator_coding",
                        "schema": indicator_coding_schema,
                        "strict": True
                    }
                }
            )

            result = json.loads(response.output_text)

            boundary_flags = result["boundary_flags"]

            # Defensive correction: if somehow 0 is mixed with other flags, remove 0.
            if "0" in boundary_flags and len(boundary_flags) > 1:
                boundary_flags = [flag for flag in boundary_flags if flag != "0"]

            return {
                "llm_primary_category": result["primary_category"],
                "llm_primary_category_label": result["primary_category_label"],
                "llm_boundary_flags": "; ".join(boundary_flags),
                "llm_confidence": result["confidence"],
                "llm_reasoning_summary": result["reasoning_summary"],
                "llm_human_review_needed": result["human_review_needed"],
                "llm_error": ""
            }

        except Exception as error:
            print(f"Attempt {attempt + 1} failed for indicator_id={row.get('indicator_id', '')}")
            print(f"Error: {error}")
            time.sleep(sleep_seconds)

    return {
        "llm_primary_category": "",
        "llm_primary_category_label": "",
        "llm_boundary_flags": "",
        "llm_confidence": 0,
        "llm_reasoning_summary": "LLM classification failed after retries.",
        "llm_human_review_needed": True,
        "llm_error": "Classification failed after retries."
    }

## Stage 12. Run LLM coding

This stage classifies all indicators from `Step1_Input_Indicators`.

The result is stored in `llm_output_df`.

This step may take several minutes if there are many indicators.

In [19]:
# ============================================================
# Stage 12. Run LLM coding
# ============================================================

results = []

for idx, row in indicators_df.iterrows():
    print(f"Coding {idx + 1} of {len(indicators_df)}: {row['indicator']}")
    
    result = classify_indicator_with_llm(row)
    results.append(result)

llm_results_df = pd.DataFrame(results)

llm_output_df = pd.concat(
    [indicators_df.reset_index(drop=True), llm_results_df.reset_index(drop=True)],
    axis=1
)

print("LLM coding completed.")
display(llm_output_df.head())

Coding 1 of 1320: Domestic comprehensive wealth index | Total | Aggregation: per capita | Not Applicable
Coding 2 of 1320: Domestic comprehensive wealth index | Total | Aggregation: total | Not Applicable
Coding 3 of 1320: Foreign assets | Total | Aggregation: per capita | Not Applicable
Coding 4 of 1320: Foreign assets | Total | Aggregation: total | Not Applicable
Coding 5 of 1320: Foreign liabilities | Total | Aggregation: per capita | Not Applicable
Coding 6 of 1320: Foreign liabilities | Total | Aggregation: total | Not Applicable
Coding 7 of 1320: Human capital | Female | Aggregation: per capita | Not Applicable
Coding 8 of 1320: Human capital | Female | Aggregation: total | Not Applicable
Coding 9 of 1320: Human capital | Male | Aggregation: per capita | Not Applicable
Coding 10 of 1320: Human capital | Male | Aggregation: total | Not Applicable
Coding 11 of 1320: Human capital | Total | Aggregation: per capita | Not Applicable
Coding 12 of 1320: Human capital | Total | Aggregati

,no,indicator,Definition,unit,source,original,dataset,llm_primary_category,llm_primary_category_label,llm_boundary_flags,llm_confidence,llm_reasoning_summary,llm_human_review_needed,llm_error
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,S,Stock,MS-NMS,0.86,The indicator measures comprehensive wealth pe...,False,
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,S,Stock,MS-NMS,0.86,The indicator measures total domestic comprehe...,False,
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,S,Stock,MS-NMS,0.90,Foreign assets are assets that persist over ti...,False,
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,S,Stock,MS-NMS,0.90,Foreign assets measured in US dollars represen...,False,
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,S,Stock,MS-NMS,0.90,Foreign liabilities are an accumulated financi...,False,


## Stage 13. Add quality checks

This stage adds simple quality checks based on the codebook.

The checks do not change the LLM output. They only add warning notes.

Examples:

- Action and Policy Guide should normally have boundary flag `0`.
- Stock and Flow should normally have one or more substantive boundary flags.
- Low-confidence cases are flagged for review.

In [20]:
def check_boundary_consistency(row):
    """
    Check whether the primary category and boundary flags are consistent with the codebook.
    """
    category = str(row.get("llm_primary_category", "")).strip()
    flags = str(row.get("llm_boundary_flags", "")).strip()
    
    if category in ["A", "P"] and flags != "0":
        return "Warning: Action or Policy Guide should normally have boundary flag 0."
    
    if category in ["S", "F"] and (flags == "0" or flags == ""):
        return "Warning: Stock or Flow should normally have at least one substantive boundary flag."
    
    if category not in ["S", "F", "A", "P"]:
        return "Warning: Invalid primary category."
    
    allowed_flags = {"MS-NMS", "DDP", "ADE", "HPS", "0"}
    parsed_flags = [flag.strip() for flag in flags.split(";") if flag.strip() != ""]
    invalid_flags = [flag for flag in parsed_flags if flag not in allowed_flags]
    
    if invalid_flags:
        return f"Warning: Invalid boundary flag: {', '.join(invalid_flags)}"
    
    return ""


llm_output_df["quality_check_note"] = llm_output_df.apply(check_boundary_consistency, axis=1)

llm_output_df["review_priority"] = llm_output_df.apply(
    lambda row: "High"
    if (
        row["llm_human_review_needed"] is True
        or float(row["llm_confidence"]) < 0.75
        or str(row["quality_check_note"]).strip() != ""
    )
    else "Normal",
    axis=1
)

print("Primary category counts:")
display(llm_output_df["llm_primary_category"].value_counts(dropna=False))

print("Boundary flag counts:")
display(llm_output_df["llm_boundary_flags"].value_counts(dropna=False))

print("Review priority counts:")
display(llm_output_df["review_priority"].value_counts(dropna=False))

display(llm_output_df.head())

Primary category counts:


llm_primary_category
P    669
F    488
S    148
A     15
Name: count, dtype: int64

Boundary flag counts:


llm_boundary_flags
0                   701
DDP                 280
MS-NMS              164
MS-NMS; DDP          70
MS-NMS; ADE          27
HPS                  26
ADE                  21
DDP; HPS              7
DDP; MS-NMS           7
MS-NMS; DDP; ADE      6
MS-NMS; HPS           5
ADE; HPS              3
MS-NMS; DDP; HPS      2
DDP; ADE              1
Name: count, dtype: int64

Review priority counts:


review_priority
Normal    828
High      492
Name: count, dtype: int64

,no,indicator,Definition,unit,source,original,dataset,llm_primary_category,llm_primary_category_label,llm_boundary_flags,llm_confidence,llm_reasoning_summary,llm_human_review_needed,llm_error,quality_check_note,review_priority
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,S,Stock,MS-NMS,0.86,The indicator measures comprehensive wealth pe...,False,,,Normal
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,S,Stock,MS-NMS,0.86,The indicator measures total domestic comprehe...,False,,,Normal
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,S,Stock,MS-NMS,0.90,Foreign assets are assets that persist over ti...,False,,,Normal
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,S,Stock,MS-NMS,0.90,Foreign assets measured in US dollars represen...,False,,,Normal
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,S,Stock,MS-NMS,0.90,Foreign liabilities are an accumulated financi...,False,,,Normal


## Stage 14. Reorder output columns

This stage organizes the output columns so that the original indicator information appears first, followed by LLM coding results and quality checks.

In [21]:
# ============================================================
# Stage 14. Reorder output columns
# ============================================================

output_columns = [
    "no",
    "indicator",
    "Definition",
    "unit",
    "source",
    "original",
    "dataset",
    "llm_primary_category",
    "llm_primary_category_label",
    "llm_boundary_flags",
    "llm_confidence",
    "llm_reasoning_summary",
    "llm_human_review_needed",
    "quality_check_note",
    "review_priority",
    "llm_error"
]

# Keep only columns that exist
output_columns = [col for col in output_columns if col in llm_output_df.columns]

llm_output_df = llm_output_df[output_columns].copy()

display(llm_output_df.head())

,no,indicator,Definition,unit,source,original,dataset,llm_primary_category,llm_primary_category_label,llm_boundary_flags,llm_confidence,llm_reasoning_summary,llm_human_review_needed,quality_check_note,review_priority,llm_error
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,S,Stock,MS-NMS,0.86,The indicator measures comprehensive wealth pe...,False,,Normal,
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,S,Stock,MS-NMS,0.86,The indicator measures total domestic comprehe...,False,,Normal,
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,S,Stock,MS-NMS,0.90,Foreign assets are assets that persist over ti...,False,,Normal,
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,S,Stock,MS-NMS,0.90,Foreign assets measured in US dollars represen...,False,,Normal,
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,S,Stock,MS-NMS,0.90,Foreign liabilities are an accumulated financi...,False,,Normal,


## Stage 15. Write results to Step4_LLM_Output

This stage creates or replaces the sheet `Step4_LLM_Output` in the same workbook.

The workflow stops here.

In [22]:
def write_df_to_existing_workbook(workbook_path, sheet_name, df):
    """
    Write a dataframe to an existing Excel workbook.
    If the sheet already exists, it is replaced.
    """
    workbook = load_workbook(workbook_path)
    
    if sheet_name in workbook.sheetnames:
        del workbook[sheet_name]
    
    workbook.save(workbook_path)
    
    with pd.ExcelWriter(
        workbook_path,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)


write_df_to_existing_workbook(
    workbook_path=WORKBOOK_PATH,
    sheet_name=OUTPUT_SHEET,
    df=llm_output_df
)

print(f"Done. LLM output has been written to sheet: {OUTPUT_SHEET}")
print(f"Workbook updated: {WORKBOOK_PATH}")

Done. LLM output has been written to sheet: Step4_LLM_Output
Workbook updated: LLM_indicator_coding_workflow.xlsx


## Stage 16. Confirm saved output

This final stage confirms that `Step4_LLM_Output` exists and displays the saved output.

No further coding or analysis is performed after this stage.

In [23]:
check_workbook = load_workbook(WORKBOOK_PATH, read_only=True)

print("Sheets in workbook:")
for sheet in check_workbook.sheetnames:
    print(" -", sheet)

if OUTPUT_SHEET in check_workbook.sheetnames:
    print(f"\nConfirmed: {OUTPUT_SHEET} exists.")
else:
    print(f"\nWarning: {OUTPUT_SHEET} was not found.")

saved_output = pd.read_excel(WORKBOOK_PATH, sheet_name=OUTPUT_SHEET)

print("\nSaved output preview:")
display(saved_output.head())

print("\nWorkflow complete. The notebook stops after Step4_LLM_Output.")

Sheets in workbook:
 - README_Workflow
 - Step1_Input_Indicators
 - Step2_Codebook
 - Step3_Examples_Gold_Set
 - Step4_LLM_Output

Confirmed: Step4_LLM_Output exists.

Saved output preview:


,no,indicator,Definition,unit,source,original,dataset,llm_primary_category,llm_primary_category_label,llm_boundary_flags,llm_confidence,llm_reasoning_summary,llm_human_review_needed,quality_check_note,review_priority,llm_error
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,S,Stock,MS-NMS,0.86,The indicator measures comprehensive wealth pe...,False,NaN,Normal,NaN
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,S,Stock,MS-NMS,0.86,The indicator measures total domestic comprehe...,False,NaN,Normal,NaN
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,S,Stock,MS-NMS,0.90,Foreign assets are assets that persist over ti...,False,NaN,Normal,NaN
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,S,Stock,MS-NMS,0.90,Foreign assets measured in US dollars represen...,False,NaN,Normal,NaN
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,S,Stock,MS-NMS,0.90,Foreign liabilities are an accumulated financi...,False,NaN,Normal,NaN



Workflow complete. The notebook stops after Step4_LLM_Output.
